In [14]:
import importlib
from pathlib import Path
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

import torch
import transformers
from transformers import AutoTokenizer, AutoModel
import numpy as np
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast

from model import GenderChunkedClassifier
from gender_dataset import GenderDataChunkedDataset, worker_init_fn, collate_fn

In [15]:
tokenizer = AutoTokenizer.from_pretrained('AIRI-Institute/gena-lm-bert-base-t2t')
model = AutoModel.from_pretrained('AIRI-Institute/gena-lm-bert-base-t2t', trust_remote_code=True)
model_module = importlib.import_module(model.__class__.__module__)
cls = getattr(model_module, 'BertModel')
model = cls.from_pretrained('AIRI-Institute/gena-lm-bert-base-t2t', add_pooling_layer=False)

/home/jovyan/chepurova/dnalm/downstream_tasks/gender_classification/venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [16]:
gender_model = GenderChunkedClassifier(model)

In [17]:
from safetensors.torch import load_model
load_model(gender_model,
           'runs/16x3072_bs_128_lr_1e-05_chrY_with_SNPs/run_1/checkpoint-98250/model.safetensors',
           )


(set(), [])

In [18]:
data_path = Path('/home/jovyan/data/downstream_tasks/gender_classification/mouse_gender_data/')
n_chunks = 16
chunk_size = 3072
seed = 1242
bs = 32

valid_dataset = GenderDataChunkedDataset(data_path / 'valid.h5', data_path / 'valid.csv',
                                         n_chunks=n_chunks, chunk_size=chunk_size,
                                         force_sampling_from_y=False,
                                         chrY_name='chrY', label_column='gender', sample_column='strain_name',
                                         seed=seed+1)

def preprocess_collate_fn(samples):
    batch = collate_fn(samples)

    batch['chunks'] = np.array(batch['chunks'])
    shape = batch['chunks'].shape
    batch['chunks'] = list(batch['chunks'].flatten())

    tokenized_batch = tokenizer(batch['chunks'], padding='longest', max_length=512,
                                truncation=True, return_tensors='pt')

    for k in tokenized_batch:
        tokenized_batch[k] = tokenized_batch[k].reshape(*shape, -1)

    batch['labels'] = torch.Tensor(batch['labels'])

    return {
        'input_ids': tokenized_batch['input_ids'],
        'attention_mask': tokenized_batch['attention_mask'],
        'labels': batch['labels'],
        'sample_ids': batch['sample_id'],
        'has_y_chr_sampled': batch['has_y_chr_sampled']
        }

valid_dataloader = DataLoader(valid_dataset, batch_size=bs, collate_fn=preprocess_collate_fn,
                          num_workers=2, pin_memory=True,
                          worker_init_fn=worker_init_fn)

In [19]:
for elem in valid_dataset:
    for chunk in elem['chunks']:
        print("N" in chunk)
    break

{'labels': 0, 'sample_id': 'C57L_J', 'sampled_chromosomes': ['J_chr5', 'J_chr8', 'J_chr15', 'J_chr10', 'J_chr14', 'J_chr6', 'J_chr9', 'J_chr9', 'J_chr1', 'J_chr9', 'J_chr9', 'J_chr11', 'J_chr4', 'J_chr18', 'J_chr2', 'J_chr16'], 'chunks': ['...'], 'full_sample_has_y_chr': True, 'has_y_chr_sampled': False}
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False


In [ ]:
gender_model = gender_model.cuda()

## no chrY upsampling

In [ ]:
dataloader = valid_dataloader

from tqdm.auto import tqdm
# N = 100000 // bs
N = 100000 // bs

labels = []
probs = []
sample_ids_list = []

for b in tqdm(dataloader, total=N):
    for k in b:
        if k not in ['sample_ids', 'has_y_chr_sampled']:
            b[k] = b[k].cuda()

    with autocast(dtype=torch.bfloat16):
        with torch.no_grad():
            outputs = gender_model(b['input_ids'], b['attention_mask'])
            b['labels'] = list(b['labels'].cpu().numpy())
            outputs['predictions'] = list(outputs['predictions'].float().cpu().numpy())
            # filter samples with chunks from chrY (if any)
            for p, label, sampleid in zip(outputs['predictions'], b['labels'], b['sample_ids']):
                labels += [label]
                probs += [p]
                sample_ids_list += [sampleid]
    N -= 1
    if N <= 0:
        break
labels = np.array(labels)
probs = np.array(probs)
sample_ids_labels = (valid_dataset.labels.loc[list(set(sample_ids_list))]['gender']).to_dict()

sample_ids_probs = {}
for i, p in zip(sample_ids_list, probs):
    if i in sample_ids_probs:
        sample_ids_probs[i] += [p]
    else:
        sample_ids_probs[i] = [p]

for k in sample_ids_probs:
    sample_ids_probs[k] = np.array(sample_ids_probs[k])
sample_ids_probs

for k in sample_ids_probs:
    probs = sample_ids_probs[k]
    print(f'{k}: [{sample_ids_labels[k]}] {(probs>0.5).sum() / len(probs):.2f}')

In [ ]:
for k in sample_ids_probs:
    probs = sample_ids_probs[k] 
    print(f'{k:20s}: [{sample_ids_labels[k]}] {(probs>1e-1).sum()}/{len(probs)} = {(probs>1e-1).sum() / len(probs):.3f}')

In [ ]:
import pickle
pickle.dump({'sample_ids_probs': sample_ids_probs, 'sample_ids_labels': sample_ids_labels},
            open(f'valid_100k_humanYsnps_on_mouse_{seed}.pckl', 'wb'))
d = pickle.load(open(f'valid_100k_humanYsnps_on_mouse_{seed}.pckl', 'rb'))

sample_ids_probs = d['sample_ids_probs']
sample_ids_labels = d['sample_ids_labels']

sample_ids_sorted = [el[0] for el in sorted(sample_ids_labels.items(), key=lambda x: x[1])]

thr = 0.5039062
thr = 0.300
for k in sample_ids_sorted:
    probs = sample_ids_probs[k]
    print(f"{k}: [{sample_ids_labels[k]}] {(probs>thr).sum()}/{len(probs)} = {(probs>thr).sum() / len(probs):.3f}")

In [ ]:
# combine all generated data

dumps = [
       'valid_100k_humanYsnps_on_mouse_1242.pckl',
#         'test_100k_humanYsnps_on_mouse.pckl',
         'test_100k_humanYsnps_on_mouse_1242.pckl'
         ]

sample_ids_probs = None
sample_ids_labels = None
for dump in dumps:

    d = pickle.load(open(dump, 'rb'))
    if sample_ids_labels is None:
        sample_ids_labels = d['sample_ids_labels']

    if sample_ids_probs is None:
        sample_ids_probs = d['sample_ids_probs']        
    else:
        sample_ids_labels.update(d['sample_ids_labels'])
        for k in d['sample_ids_probs']:
            if k in sample_ids_probs:
                sample_ids_probs[k] = np.concatenate([sample_ids_probs[k], d['sample_ids_probs'][k]])
            else:
                sample_ids_probs[k] = d['sample_ids_probs'][k]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

sample_ids_sorted = [el[0] for el in sorted(sample_ids_labels.items(), key=lambda x: x[1])]

thr = 0.3

ratios = []
for k in sample_ids_sorted:
    probs = sample_ids_probs[k]
    ratios += [(probs>thr).sum() / len(probs)]

ratios = np.array(ratios)

acc = 0
for k, r in zip(sample_ids_sorted, ratios):
    acc += int(int(r > np.median(ratios)) == sample_ids_labels[k])
acc = acc / len(sample_ids_sorted)
print(f'accuracy: {acc:0.3f}, thr: {np.median(ratios):0.3f}')


for k in sample_ids_sorted:
    probs = sample_ids_probs[k]
    r = (probs>thr).sum()/len(probs)
    print(f"{k:15s}: [{sample_ids_labels[k]}] {(probs>thr).sum()}/{len(probs)} = {r:.3f} pred: [{int(r<np.median(ratios))}]")

Ns = [25, 50, 100, 200, 500, 1000, 5000, 8000, 15000, 30000]
# thr = 0.3
accs = []
stds = []


for N in Ns:
    N_accs = []
    for i in range(200):
        ratios = []
        for k in sample_ids_sorted:
            probs = sample_ids_probs[k]
            probs = np.random.choice(probs[:,0],size=min(N, len(probs)))
            ratios += [(probs>thr).sum() / len(probs)]
            # print(f"{k}: [{sample_ids_labels[k]}] {(probs>thr).sum()}/{len(probs)} = {(probs>thr).sum() / len(probs):.3f}")

        ratios = np.array(ratios)

        acc = 0
        for k, r in zip(sample_ids_sorted, ratios):
            acc += int(int(r > np.median(ratios)) == sample_ids_labels[k])
        acc = acc / len(sample_ids_sorted)
        N_accs += [acc]
    accs += [np.mean(N_accs)]
    stds.append(np.std(N_accs))

# Creating the plot
plt.figure(figsize=(10, 6))
# plt.plot(Ns, accs, marker='o', linestyle='-', color='b')
plt.errorbar(Ns, accs, yerr=stds, marker='o', linestyle='-', color='b', capsize=5)
plt.hlines(np.mean(list(sample_ids_labels.values())), xmin=min(Ns), xmax=max(Ns), color='black', linestyle='dotted')
plt.title(f'Accuracy vs. Number of sampled subsequences (Nx16x3072bp)')
plt.xlabel('Number of sampled subsequences Nx16x3072bp')
plt.ylabel('Accuracy')
plt.grid(True)
plt.xscale('log')  # Optional: Use logarithmic scale for better visualization if needed
plt.xticks(Ns, labels=[str(n) for n in Ns])  # Ensure that all N values are labeled
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Prepare DataFrame for plotting
data_list = []
for sample_id, probs in sample_ids_probs.items():
    for prob in probs:
        data_list.append({
            'Sample ID': sample_id,  # Convert to string for consistent key usage
            'Probability': prob[0],
            'Label': sample_ids_labels[sample_id]
        })

df = pd.DataFrame(data_list)

# Sort data by label for plotting
label_order = sorted(df['Label'].unique())
sample_order = df.sort_values(by='Label')['Sample ID'].unique()

# Plotting
plt.figure(figsize=(20, 10))  # Adjusted size for better visibility
ax = sns.violinplot(x='Sample ID', y='Probability', hue='Label', data=df, inner='box',
                    order=sample_order, hue_order=label_order)
# sns.stripplot(x='Sample ID', y='Probability', data=df, color='black', size=1, jitter=True, order=sample_order)
plt.title('Sex: 0 - M, 1 - F')
plt.xticks(rotation=90)
plt.xlabel('Sample ID')
plt.ylabel('Probability')
plt.legend(title='Label')
plt.tight_layout()
plt.grid()
plt.show()